In [1]:
# This is the major code packaged as several functions. Run before running anything else.

# A function to specifically merge two ELISA files from the Omega Machine.
def ElisaMerge(filepath1,filepath2,newfilepath):
    # Files at Filepath 1 and 2 are going to be merged at the location of the newfilepath using the specific ELISA spacer between them.
    # Opening the two files that are going to be merged
    with open(filepath1, errors='replace') as f:
        lines1 = f.readlines()
    with open(filepath2, errors='replace') as f:
        lines2 = f.readlines()

    # Defining the spaces that need to be added to each row in order to get the right layout in the merged file.
    lines=[]
    spacer=[
        ",,,,,,,,,,,,",
        ",,,,,,,,,,,,",
        "",
        ",,,,,,,,,,,",
        ",,,,,,,,,,,,,,",
        "",
        ",,,,,,,,,,,,,,",
        "",
        ",",
    ]
    #Iterating across all relevant rows and merging them.
    for i in range(0,29):
        # Spacing between the tables
        if i in range(9,17) or i in range(21,29):
            lines.append(lines1[i][:-1]+",,"+lines2[i])
        # Spacing for a line in between the tables.
        elif i == 20:
            lines.append(lines1[i][:-1]+","+lines2[i])
        # Spacing for the info in between the tables.
        elif i in range(17,20):
            lines.append(lines1[i][:-1]+",,,,,,,,,,,,,,"+lines2[i])
        # Specific Spacers for all the other 
        else:
            lines.append(lines1[i][:-1]+spacer[i]+lines2[i])

    with open(newfilepath, "w") as f:
        pass
        f.writelines(lines)


# A function to merge all ELISA files in a folder
from pathlib import Path
import openpyxl
import os

def MergeAllInFolder(folderpath):
    # This function merges all the CSV files in a folder in alphabetical order and returns the path of the combined Excel file.
    # Replace this with your folder path
    folderpath=folderpath+"/"

    # Defining the path where the combined file is saved.
    LastFolderName = folderpath.split("/")[-2]
    combinedname= LastFolderName
    combinepath = folderpath+combinedname+".csv"
    combinedpathXLSX = folderpath+combinedname+".xlsx"

    # Get all CSV files and sort alphabetically
    folder_path = Path(folderpath)
    csv_files = sorted(folder_path.glob("*.CSV"))

    #
    filepath=folderpath+csv_files[0].name
    with open(filepath, errors='replace') as f:
        lines1 = f.readlines()

    with open(combinepath, "w") as f:
        pass
        f.writelines(lines1)

    for i in range(1,len(csv_files)):
        filepath=folderpath+csv_files[i].name
        ElisaMerge(combinepath,filepath,combinepath)

    ## Save the file as Excel
    # Creating a new empty Excel workbook
    wb = openpyxl.Workbook()
    ws = wb.active

    # Making sure all numbers are saved as numbers in the Excel and appending the empty Excel sheet with that.
    with open(combinepath) as f:
        reader = f.readlines()
        for row in reader:
            row = row.split(",")
            for i in range(0,len(row)):
                try:
                    row[i]=float(row[i])
                except:
                    row[i]=row[i]
            ws.append(row)


    wb.save(combinedpathXLSX)

    # Delete the combined.csv
    os.remove(combinepath)

    print("File can be found here:")
    print(combinedpathXLSX)
    print("")
    print("Done with merging!")
    print("")
    return combinedpathXLSX

# A function to color the tables in that file.
import openpyxl as xl
from openpyxl.formatting.rule import CellIsRule, ColorScaleRule, FormulaRule

def ColorTheTables(path):
    #Conditionally formatting the 450 and 570 tables.
    
    # Open the excelfile from the previous block and select the active sheet
    wb = xl.load_workbook(path)
    ws = wb.active

    # Coloring of the 450 tables
    # Create a rule: High values red, low values green, yellow in between
    color_scale_rule_450 = ColorScaleRule(start_type='percentile', start_value=1, start_color='63BE7B',
                                          mid_type='percentile', mid_value=50, mid_color='FFEB84',
                                          end_type='percentile', end_value=99, end_color='F8696B')
    
    # Iterate over all the tables and apply the rule
    for i in range(0,int(ws.max_column/14)):
        start = ws.cell(row=10, column=2+14*i).coordinate
        end = ws.cell(row=17, column=13+14*i).coordinate
        table = start+":"+end
        ws.conditional_formatting.add(table, color_scale_rule_450)
    
    # Coloring the 570 Tables
    # Create a rule: High values red, low values blue, white in between
    color_scale_rule_570 = ColorScaleRule(start_type='percentile', start_value=1, start_color='638ABE',
                                          mid_type='percentile', mid_value=50, mid_color='FFFFFF',
                                          end_type='percentile', end_value=99, end_color='F8696B')
    
    # Iterate over all the tables and apply the rule
    for i in range(0,int(ws.max_column/14)):
        start = ws.cell(row=22, column=2+14*i).coordinate
        end = ws.cell(row=29, column=13+14*i).coordinate
        table = start+":"+end
        ws.conditional_formatting.add(table, color_scale_rule_570)
    
    
    # Save the workbook
    wb.save(path)
    
    print("Done with coloring!")
    print("")

# A function to rename the columns of the samples in that file.
import openpyxl as xl
from copy import copy

def RenameTheColumns(path, Npath):
    # Rename the columns after what is actually in them.
    # Open the Excel file from the previous block with the data and select the active sheet
    wb = xl.load_workbook(path)
    ws = wb.active
    Nwb = xl.load_workbook(Npath)
    names = Nwb["Names"]
    
    for i in range(0,names.max_row):
        for j in range(1,13):
            # Get the value and fill from the Names Excel
            value = names.cell(row=i+1,column=j+1).value
            fill = names.cell(row=i+1,column=j+1).fill
            # Paste them on the 450 table
            ws.cell(row=8,column=i*14+j+1).value= value
            ws.cell(row=8,column=i*14+j+1).fill = copy(fill)
            
    
    wb.save(path)
    
    print("Done with adding the sample names!")
    print("")

# Function to get dilution information from the metainfo Excel
import openpyxl as xl

def GetDilInfo(mergerpath, Mpath):
    path = mergerpath
    Mpath = Mpath
    
    # Loading the dilution information from the Metadata Excel
    wb = xl.load_workbook(path)
    ws = wb.active
    Nwb = xl.load_workbook(Mpath)
    dilutions = Nwb["Dilutions"]
    standard_start = dilutions["B2"].value
    standard_dil = dilutions["B3"].value
    sample_start = dilutions["B6"].value
    sample_dil = dilutions["B7"].value
    
    # Calculating the dilution series based on that data
    seriesST = [standard_start / (standard_dil ** i) for i in range(8)]
    seriesSA = [sample_start * (sample_dil ** i) for i in range(8)]
    
    #print("Standard Starting Concentration : ",standard_start)
    #print(" and Dilution Coefficient       : ",standard_dil)
    #print(" and Dilution Con Series        : ",seriesST)
    #print("")
    #print("Sample Starting Dilution Factor : ",sample_start)
    #print(" and Dilution Coefficient       : ", sample_dil)
    #print(" and Dilution Factor Series     : ",seriesSA)
    #print("")

    dilutions = [seriesST, seriesSA]

    return dilutions

# Function taking the start of the merged file and creating dict with all columns as lists.
def GetSeries(tablenumber,ws):
    start=(tablenumber)*14
    results=dict()
    for col in range(start+2,start+14):
        name =str(ws.cell(row=8,column=col).value)
        if name == "blank":
            name = "blank"+str(col)
        results[name] = list()
        for el in list(ws.columns)[col-1][9:17]:
            results[name].append(el.value)
    return results

# Function to plot all the data from one plate
import matplotlib.pyplot as plt
import numpy as np
import os

def PlotTable(results,tablenumber):
    legend=[]
    fig, axs = plt.subplots(1,2,figsize=(12, 12/3))
    for series in results:
        if series.startswith("blank"):
            axs[0].semilogx(seriesSA,results[series])
            legend.append(series)
        elif series == "standard":
            axs[1].semilogx(seriesST,results[series])
        else:
            axs[0].semilogx(seriesSA,results[series])
            legend.append(series)
    axs[0].legend(legend)
    axs[1].legend(["standard"])
    title = "Data from "+ws.cell(row=4,column=tablenumber*14+3).value[5:]
    fig.suptitle(title)
    plt.show()

# Function to calculate sigmoidal curve fit
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

def sigFit(xdata,ydata):
    def four_pl(x, bottom, top, ic50, hill):
        return bottom + (top - bottom) / (1 + (x / ic50)**hill)

    # initial guesses
    p0 = [0.03, 3, 50, 1]

    # bounds to prevent fitting errors
    bounds = (
        [0.029, 0.5, 0, -1],     
        [0.031, 5, 1e6, 1]
    )

    params, cov = curve_fit(four_pl, xdata, ydata, p0=p0, bounds=bounds, maxfev=10000)

    bottom, top, ic50, hill = params

    return params

#Function to calculate EC50s for a single table
def calculateEC50s(datafrom450, dilutions):
    results = datafrom450
    dil=dilutions

    ic50s = []
    ic50s.append("Est. ED50")
    for el in results:
        if el == "standard":
            ic50s.append("/")
        elif el.startswith("blank"):
            ic50s.append("/")
        else:
            ic50 = round(sigFit(dil[1], results[el])[2])
            if ic50 < 10:
                ic50 = 10
            ic50s.append(ic50)
    ic50s.append("")
    return ic50s

# Function to add EC50 data to an Excel File as the last row.
def addEC50(mergerpath,dil):
    path = mergerpath
    wb = xl.load_workbook(path)
    ws = wb.active
    totalEC50 = []
    for i in range(int(ws.max_column/14)):
        results = GetSeries(i,ws)
        tableEC50 = calculateEC50s(results,dil)
        for el in tableEC50:
            totalEC50.append(el)
        #PlotTable(results,i)
        
    ws.append([])
    ws.append(totalEC50)
    color_scale_rule = ColorScaleRule(start_type='percentile', start_value=1, start_color='63BE7B',
                                      mid_type='percentile', mid_value=50, mid_color='FFEB84',
                                      end_type='percentile', end_value=99, end_color='F8696B')
    for i in range(0,int(ws.max_column/14)):
        start = ws.cell(row=ws.max_row, column=3+14*i).coordinate
        end = ws.cell(row=ws.max_row, column=12+14*i).coordinate
        table = start+":"+end
        ws.conditional_formatting.add(table, color_scale_rule)
    wb.save(path)

    print("Done with estimating EC50s!")
    print("")

# Inverse sigmoidal function
def inverse_4pl(y, top, bottom, hill, ic50):
    y = np.asarray(y)
    eps = 1e-12
    y = np.clip(y, bottom + eps, top - eps)
    return ic50 * ((top - y) / (y - bottom)) ** (1.0 / hill)

# Function for calculating concentrations
def addCon(mergerpath,dil):
    path = mergerpath
    wb = xl.load_workbook(path)
    ws = wb.active
    
    conRow = []
    choValRow = []
    choDilRow = []
    
    for i in range(int(ws.max_column/14)):
        results = GetSeries(i,ws)
    
        STindex = 0
        while max(results["standard"]) > 2:
            results["standard"].remove(max(results["standard"]))
            STindex = STindex + 1
        
        params = sigFit(dil[0][STindex:],results["standard"])
        bottom, top, ic50, hill = params
        totalCon = []
        chosenVal = []
        chosenDil = []
        for el in results:
            if el.startswith("blank"):
                totalCon.append("/")
                chosenVal.append("/")
                chosenDil.append("/")
            elif el == "standard":
                first = results[el][0]
                con = inverse_4pl(first, top, bottom, hill, ic50)*5**STindex/1000
                totalCon.append(con)
                chosenVal.append(first)
                chosenDil.append(str(dil[0][STindex])+" (Con)")
            else:
                indexDil=0
                if results == []:
                    results = [1.49]
                while max(results[el]) > 1.5:
                    results[el].remove(max(results[el]))
                    indexDil = indexDil + 1
                    if results[el] == []:
                        results[el] = [1.49]
                        indexDil = indexDil-1
                con = inverse_4pl(max(results[el]), top, bottom, hill, ic50)*dil[1][indexDil]
                con = round(con/1000,2)
                totalCon.append(con)
                chosenVal.append(max(results[el]))
                chosenDil.append(dil[1][indexDil])
        
        conRow.append("Interp. Con.")
        for el in totalCon:
            conRow.append(el)
        conRow.append("")
        
        choValRow.append("Chos. Val.")
        for el in chosenVal:
            choValRow.append(el)
        choValRow.append("")
        
        choDilRow.append("Chos. Dil.")
        for el in chosenDil:
            choDilRow.append(el)
        choDilRow.append("")
    
    ws.append([])
    ws.append(choValRow)
    ws.append(choDilRow)
    ws.append(conRow)
    
    color_scale_rule = ColorScaleRule(start_type='percentile', start_value=1, start_color='63BE7B',
                                      mid_type='percentile', mid_value=50, mid_color='FFEB84',
                                      end_type='percentile', end_value=99, end_color='F8696B')
    for i in range(0,int(ws.max_column/14)):
        start = ws.cell(row=ws.max_row, column=3+14*i).coordinate
        end = ws.cell(row=ws.max_row, column=12+14*i).coordinate
        table = start+":"+end
        ws.conditional_formatting.add(table, color_scale_rule)
    
    wb.save(path)

    print("Done with estimating Concentrations based on interpolated standard curve!")
    print("")

In [4]:
# TODOs:
#  - Change the folder path to your folder path. (You can copy paths on mac by selecting the folder and pressing cmd+opt+C.)
#  - Add a Metadata Excel File according the the template and adjust the names and dilution sheet. Put it into the same folder or adjust Npath.
folderpath = '/Users/kasimirreich/Documents/Nussenzweig Lab/Data/ELISA/260521_invivo2_yu2gp140_anti101074_d-48,d0,d7,d14'
Mpath = folderpath+'/MetaData.xlsx'

# Merge the files
mergerpath = MergeAllInFolder(folderpath)
# Color the tables
ColorTheTables(mergerpath)
# Add the titles to the file 
RenameTheColumns(mergerpath, Mpath)
# Get dilution infos
dil = GetDilInfo(mergerpath, Mpath)
# Add the EC50 estimates
addEC50(mergerpath,dil)
# Add the estimated concentrations
addCon(mergerpath,dil)

File can be found here:
/Users/kasimirreich/Documents/Nussenzweig Lab/Data/ELISA/260521_invivo2_yu2gp140_anti101074_d-48,d0,d7,d14/260521_invivo2_yu2gp140_anti101074_d-48,d0,d7,d14.xlsx

Done with merging!

Done with coloring!

Done with adding the sample names!

Done with estimating EC50s!

Done with estimating Concentrations based on interpolated standard curve!

